In [ ]:
import json

from mlx_vlm import generate, load
from mlx_vlm.prompt_utils import apply_chat_template

# model, processor = load("numind/NuExtract3-mlx-mxfp8")  # or the mlx-community repo
# model_4_bit, processor_4_bit = load("numind/NuExtract3-mlx-4bits")

template = {
    "store": "verbatim_string",
    "items": [{
        "name": "verbatim_string",
        "quantity": "integer",
        "unit_price": "number",
        "line_total": "number",
    }],
    "discounts": [{"description": "verbatim_string", "amount": "number"}],
    "total": "number",
}
instructions = (
    "Each item has exactly one line_total, the rightmost amount for that item. "
    "A line like '7@ €0.49 €3.43' under an item name means quantity 7, "
    "unit_price 0.49, line_total 3.43, and belongs to the item directly above it. "
    "Items without a quantity line have quantity 1 and unit_price equal to line_total. "
    "Put multi-buy deals and discounts in `discounts` as negative amounts. "
    "Ignore the VAT analysis and payment sections."
)
image = ["sv_receipt_small.jpg"]

In [ ]:
from pathlib import Path
import urllib.request

url = "https://scontent-lhr11-1.xx.fbcdn.net/v/t1.6435-9/125155019_377920886958290_414037003099732973_n.jpg?stp=dst-jpg_tt6&cstp=mx540x960&ctp=s540x960&_nc_cat=101&_nc_map=urlgen_bucketless&ccb=1-7&_nc_sid=833d8c&_nc_ohc=eFZWXxgj0aoQ7kNvwEHJ8mj&_nc_oc=AdqThIZRtlbXeZ-IJYuCONF6D_rFd9P0mev7mU2srhikg_ZndVZrCXa1ZHUNedcoHjnnkR6sKOiwjrZYogKcNLUK&_nc_zt=23&_nc_ht=scontent-lhr11-1.xx&_nc_gid=M2QZ0uSW8LYhv7P_V58xyQ&_nc_ss=7b289&oh=00_AQRmcase6P445KV3qpl6kpfH5vvsrJoD86P8djO287mCyw&oe=6B1D2FFD"
path = Path("sv_receipt.jpg")

if not path.exists():
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r:
        path.write_bytes(r.read())


from PIL import Image

img = Image.open("sv_receipt.jpg")
img.thumbnail((1536, 1536))   # keeps aspect ratio, long side max 1536px
img.save("sv_receipt_small.jpg", quality=95)

In [ ]:
formatted_prompt = apply_chat_template(
    processor, model.config,
    num_images=1,
    template=json.dumps(template),           # the structure, as a JSON string
    prompt="",          # optional
    instructions=instructions,
    enable_thinking=False,

)

out = generate(model, processor, formatted_prompt, image, temperature =0.2, max_tokens=1024)
processed_receipt = json.loads(out.text)

In [ ]:
formatted_prompt = apply_chat_template(
    processor_4_bit, model_4_bit.config,
    num_images=1,
    prompt="",
    template=json.dumps(template),           # the structure, as a JSON string
    instructions=instructions,          # optional
    enable_thinking=False,

)

out_4_bit = generate(model_4_bit, processor_4_bit, formatted_prompt, image,temperature =0.2 , max_tokens=1024)
processed_receipt_4_bit = json.loads(out_4_bit.text)

In [ ]:

for it in processed_receipt_4_bit["items"]:
    if abs(it["quantity"] * it["unit_price"] - it["line_total"]) >= 0.01 and it["unit_price"]:
        q = round(it["line_total"] / it["unit_price"])
        if abs(q * it["unit_price"] - it["line_total"]) < 0.01:
            it["quantity"] = q          # e.g. 3.43 / 0.49 = 7
        else:
            it["needs_review"] = True

In [ ]:
processed_receipt_4_bit

In [ ]:
processed_receipt

In [ ]:
processed_receipt_4_bit == processed_receipt